In [1]:
# Implementations from https://github.com/e-delaney/Instance-Based_CFE_TSC

In [2]:
cd ../

/home/gupt_ad/conclusion_work/experiments/subspace


In [3]:
import os
import sys
import pickle
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn import preprocessing
from tslearn.neighbors import KNeighborsTimeSeries
import tensorflow as tf
from tensorflow import keras

from experiments.experiment_utils import local_data_loader, label_encoder
import warnings
warnings.filterwarnings('ignore')

print(tf.__version__)

2025-11-10 12:54:05.770327: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-10 12:54:05.865202: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-10 12:54:06.146639: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-10 12:54:06.146733: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-10 12:54:06.148123: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

2025-11-10 12:54:09.341565: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

2.14.0


In [4]:
# datasets = ['CBF', 'chinatown', 'coffee', 'gunpoint', 'ECG200']
datasets = ['CBF']

In [5]:
!pwd

/home/gupt_ad/conclusion_work/experiments/subspace


# Load data and models

In [6]:
data_dict = {}
models_dict = {}
outlier_calculators_dict = {}
nuns_idx_dict = {}
desired_classes_dict = {}

for dataset in datasets:
    X_train, y_train, X_test, y_test = local_data_loader(str(dataset), data_path="./experiments/data")
    y_train, y_test = label_encoder(y_train, y_test)
    data_dict[dataset] = (X_train, y_train, X_test, y_test)

    # Load model
    model = keras.models.load_model(f'./experiments/models/{dataset}/{dataset}_best_model.hdf5')
    y_pred = np.argmax(model.predict(X_test), axis=1)
    models_dict[dataset] = model

2025-11-10 12:54:12.154510: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2211] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


29/29 [==============================] - 1s 14ms/step


# Native Guide counterfactuals

In [7]:
def native_guide_retrieval(query, predicted_label, distance, n_neighbors, X_train, y_train):
    df = pd.DataFrame(y_train, columns = ['label'])
    df.index.name = 'index'
    df[df['label'] == 1].index.values, df[df['label'] != 1].index.values
    ts_length = X_train.shape[1]
    
    knn = KNeighborsTimeSeries(n_neighbors=n_neighbors, metric = distance)
    knn.fit(X_train[list(df[df['label'] != predicted_label].index.values)])
    dist,ind = knn.kneighbors(query.reshape(1,ts_length), return_distance=True)
    
    return dist[0], df[df['label'] != predicted_label].index[ind[0][:]]

In [8]:
def findSubarray(a, k): #used to find the maximum contigious subarray of length k in the explanation weight vector
    n = len(a)
    vec=[] 

    # Iterate to find all the sub-arrays 
    for i in range(n-k+1): 
        temp=[] 
        # Store the sub-array elements in the array 
        for j in range(i,i+k): 
            temp.append(a[j]) 
        # Push the vector in the container 
        vec.append(temp) 

    sum_arr = []
    for v in vec:
        sum_arr.append(np.sum(v))

    return (vec[np.argmax(sum_arr)])

In [9]:
def counterfactual_generator_swap(instance, nun, subarray_length):
    
    most_influencial_array = findSubarray((cam_training_weights[nun]), subarray_length)
    starting_point = np.where(cam_training_weights[nun]==most_influencial_array[0])[0][0]
    X_example = np.concatenate((X_test[instance][:starting_point], (X_train[nun][starting_point:subarray_length+starting_point]), X_test[instance][subarray_length+starting_point:]))
    prob_target = model.predict(X_example.reshape(1,-1,1), verbose=0)[0][y_pred[instance]]
    
    while prob_target > 0.5:
        
        subarray_length +=1
        most_influencial_array=findSubarray((cam_training_weights[nun]), subarray_length)
        starting_point = np.where(cam_training_weights[nun]==most_influencial_array[0])[0][0]
        X_example = np.concatenate((X_test[instance][:starting_point], (X_train[nun][starting_point:subarray_length+starting_point]), X_test[instance][subarray_length+starting_point:]))
        prob_target = model.predict(X_example.reshape(1,-1,1), verbose=0)[0][y_pred[instance]]
        
    return X_example

In [ ]:
for dataset in datasets:
    print(f'Generating counterfactuals for {dataset}...')
    # Load data and model
    X_train, y_train, X_test, y_test = data_dict[dataset]
    model = models_dict[dataset]
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

    # Get the NUNs
    nuns_idx = []
    for instance_idx in range(len(X_test)):
        nuns_idx.append(native_guide_retrieval(X_test[instance_idx], y_pred[instance_idx], 'euclidean', 1, X_train, y_train)[1][0])
    nuns_idx = np.array(nuns_idx)

    # Get cam importances 
    cam_training_weights = np.load(f'./methods/NativeGuide/Class_Activation_Mapping/{dataset}_cam_train_weights.npy')
    cam_testing_weights = np.load(f'./methods/NativeGuide/Class_Activation_Mapping/{dataset}_cam_test_weights.npy')

    # Get the counterfactuals 
    ng_cfs = []
    test_instances_idx = np.array(range(len(X_test)))
    for test_instance_idx, nun_idx in tqdm(zip(test_instances_idx, nuns_idx)):
        ng_cfs.append(counterfactual_generator_swap(test_instance_idx, nun_idx, 1))

    # Store
    # Adapt counterfactual result to our format
    results = [{'cf': np.expand_dims(cf, axis=0), 'time': -1} for cf in ng_cfs]
    # Store concatenated file
    with open(f'./experiments/results/{dataset}/ng.pickle', 'wb') as f:
        pickle.dump(results, f, pickle.HIGHEST_PROTOCOL)
    # with open(f'./experiments/results/{dataset}/ng.pickle', 'wb') as f:
    #     pickle.dump(ng_cfs, f, pickle.HIGHEST_PROTOCOL)

Generating counterfactuals for CBF...


15it [00:54,  3.21s/it]

In [ ]:
ng_cfs[0].shape 

In [ ]:
results = [{'cf': np.expand_dims(cf, axis=0), 'time': -1} for cf in ng_cfs]

In [ ]:
results[0]['cf'].shape

# Watcher et al

In [ ]:
import pickle

In [ ]:
for dataset_name in datasets:
    X_train, _, _, _ = data_dict[dataset_name]
    ts_length, n_features = X_train.shape[1], X_train.shape[2]
    with open(f'./experiments/results/{dataset_name}/counterfactuals_wcf_ng.pickle', 'rb') as f:
        wcf_ng_cfs = pickle.load(f)
        
    results = [{'cf': cf.reshape(1 ,ts_length, n_features), 'time': -1} for cf in wcf_ng_cfs]
    # Store concatenated file
    with open(f'./experiments/results/{dataset_name}/wcf_ng.pickle', 'wb') as f:
        pickle.dump(results, f, pickle.HIGHEST_PROTOCOL)
        

In [ ]:
results

In [ ]:
from scipy.optimize import minimize
from scipy import stats

In [ ]:
def target_(label):
    if label == 0:
        counter = 1
    elif label == 1:
        counter = 0
    return counter

def dist_mad(query, cf):
    manhat = np.abs(query-cf)
    mad = stats.median_abs_deviation(X_train)
    return np.sum((manhat/mad).flatten())

def loss_function_mad(x_dash):
    target = target_(example_label)
    L = lamda*(model.predict(x_dash.reshape(1,-1,1), verbose=0)[0][target] - 1)**2 + \
    dist_mad(x_dash.reshape(1,-1,1), query)
    return L

In [ ]:
def Wachter_Counterfactual(instance, lambda_init):
    
    global lamda
    global dist_mad
    global loss_function_mad
    global example_label
    global query

    
    pred_threshold = 0.5

    # initial conditions
    lamda = lambda_init
    x0 = X_test[instance].reshape(1,-1,1) # initial guess for cf
    query = X_test[instance].reshape(1,-1,1)
    example_label = y_pred[instance]

    res = minimize(loss_function_mad, x0.reshape(1,-1), method='nelder-mead', options={'maxiter':10, 'xatol': 50, 'adaptive': True})
    cf = res.x.reshape(1,-1,1)

    target = target_(y_pred[instance])
    prob_target = model.predict(cf)[0][target]


    i=0
    while prob_target < pred_threshold:


        lamda = lambda_init*(1+0.5)**i
        x0 = cf
        res = minimize(loss_function_mad, x0.reshape(1,-1), method='nelder-mead', options={'maxiter':10, 'xatol': 50, 'adaptive': True})
        cf = res.x.reshape(1,-1,1)
        
        """figure = plt.Figure()
        plt.plot(cf.flatten())
        plt.show()"""
        
        prob_target = model.predict(cf, verbose=0)[0][target]
        i += 1
        if i == 500:
            print('Error condition not met after',i,'iterations')
            break
    
    return cf

In [ ]:
for dataset in datasets[4:]:
    print(f'Generating counterfactuals for {dataset}...')
    # Load data and model
    X_train, y_train, X_test, y_test = data_dict[dataset]
    model = models_dict[dataset]
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

    # Generate counterfactuals
    wcf_cfs = []
    for instance in tqdm(range(len(X_test))):
        wcf_cfs.append(Wachter_Counterfactual(instance,lambda_init=0.1)[0])
        print(wcf_cfs)
    
    # Store
    with open(f'./counterfactuals/results/{dataset}/counterfactuals_wcf_ng.pickle', 'wb') as f:
        pickle.dump(wcf_cfs, f, pickle.HIGHEST_PROTOCOL)